In [ ]:
from wildlife_datasets import analysis, datasets

In [ ]:
!uv run kaggle auth login

In [ ]:
from open_vocab_mot import WILDLIFE_10K_PATH, WILDLIFE_10K_SIDECAR_PATH
print(WILDLIFE_10K_PATH, WILDLIFE_10K_SIDECAR_PATH)

In [ ]:
datasets.WildlifeReID10k.get_data(str(WILDLIFE_10K_PATH))

In [ ]:
wildlife_10k = datasets.WildlifeReID10k(str(WILDLIFE_10K_PATH))

In [ ]:
wildlife_10k.df

In [ ]:
wildlife_10k.df["dataset"].unique()

In [ ]:
wildlife_10k.df["split"].unique()

In [ ]:
counts = wildlife_10k.df["identity"].value_counts()
valid_identities = counts[counts >= 8*3].index
restricted_df = wildlife_10k.df[wildlife_10k.df["identity"].isin(valid_identities)].copy()

In [ ]:
# restricted_df

In [ ]:
# restricted_df[restricted_df["identity"] == "ZindiTurtleRecall_t_id_m2JvEcsg"]

In [ ]:
from open_vocab_mot.data import Wildlife10KSubsetDataset, Wildlife10KSplit, Wildlife10KDatasets, VideoReIDKPFBatchIterableDataset, VideoReIDItem, collate_video_reid_ds

In [ ]:
dataset_id: Wildlife10KDatasets = "BelugaID"

In [ ]:
test_df = wildlife_10k.df[wildlife_10k.df["dataset"] == dataset_id]
# test_df = wildlife_10k.df[wildlife_10k.df["cluster_id"] == "BelugaID_whale000_0"]
test_df

In [ ]:
test_ds = Wildlife10KSubsetDataset(
    WILDLIFE_10K_PATH,
    dataset_id,
    Wildlife10KSplit.TRAIN,
    sidecar_root=WILDLIFE_10K_SIDECAR_PATH,
    load_image_pil=True,
    load_image_tensor=False,
    load_segmentations=True,
    pre_loaded_ds=wildlife_10k,
    verbose=True
)

In [ ]:
sample = test_ds[10]
sample

In [ ]:
from IPython.display import display
from torchvision.transforms.v2.functional import to_pil_image

In [ ]:
display(sample.frame)
display(to_pil_image(sample.segmentation_tensor))

In [ ]:
from torch.utils.data import DataLoader
sampler_ds = VideoReIDKPFBatchIterableDataset(
    test_ds, batches_per_epoch=100,
    num_identities_per_batch=1,
    num_sequences_per_identity=2,
    num_frames_per_sequence=2,
    allow_same_identity_same_sequence=True,
    allow_resampling_sample_indices=True
)

sampler_loader = DataLoader(sampler_ds, batch_size=None, collate_fn=collate_video_reid_ds)

In [ ]:
batch: VideoReIDItem = next(iter(sampler_loader))

In [ ]:
batch

In [ ]:
display(batch.frames[0])
display(batch.frames[1])
display(to_pil_image(batch.segmentations[0]))
display(to_pil_image(batch.segmentations[1]))

In [ ]:
display(batch.frames[2])
display(batch.frames[3])
display(to_pil_image(batch.segmentations[2]))
display(to_pil_image(batch.segmentations[3]))